In [1]:
import torch
from dinosaw.wrappers import ModelTypes, MODEL_NAMES, get_models
from dinosaw.utils import do_2D_pca, get_features, add_custom_font

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:1'
half = False

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'sinusoid_dv2', 'nope', 'alibi_dv2') 
# models = get_models(enabled_models, "../../trained_models", DEVICE, half)
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'sinusoid_dinov2_s_cb', 'nope_dinov2_s', 'alibi_coco_dinov2_s',)
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

data_dir = "../paper_figures/data/nope_vs_alibi"

imgs = ("plant_cells.jpg", "cat_sketch.jpeg", "deer_sketch.png")

2026-07-24 14:02:48 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:1
2026-07-24 14:02:48 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-07-24 14:02:48 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:1, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 14:02:48 | I | factory.py                 : 152 | Building wrapper 'sinusoid_dinov2_s_cb' on device cuda:1
2026-07-24 14:02:48 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path='../../models/checkpoints/ablations/sinusoid_dv2_cb.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[<function replace_pe_with_sincos at 0x7f7f001adee0>], dtype=torch.float32)
2026-07-24 14:02:48 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/ablations/sinusoid_dv2_cb.pth
2026-07-24 14:02:48 | I | wrapper.py                 :  48 | I

In [3]:
channel_group = 0
features, features_reduced = [], {model_key: {} for model_key in enabled_models}
for img_file in imgs:
    for model_key in enabled_models:
        model = models[model_key]
        img_path = f'{data_dir}/{img_file}'
        img = Image.open(img_path).convert('RGB')
        sf_calc = 518 / img.height
        if img_file == "plant_cells.jpg":
            sf_calc = 1
        img = img.resize((int(sf_calc * img.width), int(sf_calc * img.height)), Image.LANCZOS)
        feats = get_features(model, img, channel_last=False)#.cpu()
        features_reduced[model_key][img_file] =  do_2D_pca(feats, (channel_group+1)*3, post_norm='minmax')[:, :, channel_group*3:channel_group*3+3]

2026-07-24 14:02:49 | I | wrapper.py                 :  92 | Processing image, size: [1810, 1191]


2026-07-24 14:02:49 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,1190,1806] -> f: [1,384,85,129]
2026-07-24 14:02:49 | I | wrapper.py                 :  92 | Processing image, size: [1810, 1191]
2026-07-24 14:02:50 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,1190,1806] -> f: [1,384,85,129]
2026-07-24 14:02:50 | I | wrapper.py                 :  92 | Processing image, size: [1810, 1191]
2026-07-24 14:02:50 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,1190,1806] -> f: [1,384,85,129]
2026-07-24 14:02:50 | I | wrapper.py                 :  92 | Processing image, size: [1810, 1191]
2026-07-24 14:02:50 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,1190,1806] -> f: [1,384,85,129]
2026-07-24 14:02:51 | I | wrapper.py                 :  92 | Processing image, size: [407, 518]
2026-07-24 14:02:51 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,406] -> f: [1,384,37,29]
2026-07-24 14:02:5

In [20]:
# %%capture
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')

from matplotlib.gridspec import GridSpec

NCOLS=len(imgs)
NROWS = 1 + len(enabled_models)
# H,W = 14,19

# fig, axs = plt.subplots(nrows=NROWS, ncols=NCOLS, figsize=(25, 25), gridspec_kw={"hspace":0.5})
fig = plt.figure(figsize=(7.5, 2.3 * (3.5)))
gs = GridSpec(NROWS, NCOLS, figure=fig)



for i, img_file in enumerate(imgs):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(Image.open(f"{data_dir}/{img_file}").convert("RGB"))
    ax.set_axis_off()
    for j, model_key in enumerate(enabled_models):
        ax = fig.add_subplot(gs[j+1, i])
        feats=features_reduced[model_key][img_file]
        ax.imshow(feats,)
        ax.set_yticks([])
        ax.set_xticks([])
        if i==0:
            model_name = MODEL_NAMES[model_key]
            label = model_name.strip('(COCO)') if "alibi" in model_key.lower() else model_name.strip('(CB)')
            ax.set_ylabel(label, fontweight="bold" if "alibi" in model_key else None)
# plt.tight_layout()

SAVE = True
if SAVE:
    plt.savefig("saved/S9.pdf", dpi=300, bbox_inches='tight')
    plt.close()

findfont: Failed to find font weight normal, now using 300.
